# 15 OptiTrack Redo / Repopulation Template

Template for safe reprocessing and repopulation of OptiTrack tables.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:

keys = (
    scan.Scan * session.Session * session.SessionUser * subject.User
    & f'initials = "{INITIALS}"'
    & f'session_datetime >= "{DATE_FROM}"'
).fetch("KEY")

rows = []
for key in keys:
    rows.append(
        {
            **key,
            "optitrack_events": len(event.Event & key & 'event_type LIKE "%optitrack%"'),
            "motioncapture_rows": len(mocap.MotionCapture & key),
            "rigidmouse_rows": len(virtual_markers_optitrack.RigidMouseTracking & key),
        }
    )

redo_df = pd.DataFrame(rows)
redo_df


In [ ]:

redo_candidates = redo_df[
    (redo_df["optitrack_events"] > 0)
    & (
        (redo_df["motioncapture_rows"] == 0)
        | (redo_df["rigidmouse_rows"] == 0)
    )
]

redo_candidates


In [ ]:

if ALLOW_DB_WRITES:
    raise RuntimeError("Set ALLOW_DB_WRITES manually after explicit approval.")

# Recommended write-enabled execution order:
# 1) mocap.MotionCapture.populate(restriction, ...)
# 2) virtual_markers_optitrack.RigidMouseTracking.populate(restriction, ...)
# 3) pupil_tracking.GazeReconstruction3D.populate(restriction, ...)
